In [ ]:
# Only run this once at the beginning of the notebook to set up the correct working directory and import paths
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # because notebook is inside data_preprocessing
sys.path.append(str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f"Current working directory: {os.getcwd()}")

## Load data

In [ ]:
from data.process_raw_data import read_3DCraft_npy_file
import numpy as np


def compute_3DCraft_block_frequency(directory_path, max_size=None) -> dict: 
    house_folders = [f for f in os.listdir(directory_path) if f.startswith("workdir")]
    if max_size is not None:
        house_folders = house_folders[:max_size]
    
    block_frequency_dict = {}
    for house_folder in house_folders:
        # print(f"Processing {house_folder} for unique block types...")
        try:
            voxels, block_types = read_3DCraft_npy_file(directory_path / house_folder)
        except:
            print(f"Error processing {house_folder}")
            continue
        # Flatten voxel
        unique_indices, counts = np.unique(voxels, return_counts=True)

        for block_index, count in zip(unique_indices, counts):
            block_name = block_types.get(block_index, "UNKNOWN")
            block_frequency_dict[block_name] = (block_frequency_dict.get(block_name, 0) + int(count))
        del voxels, block_types

    return block_frequency_dict


def compute_3DCraft_block_frequency_individual(directory_path, house_folder) -> dict:     
    block_frequency_dict = {}
    try:
        voxels, block_types = read_3DCraft_npy_file(directory_path / house_folder)
    except:
        print(f"Error processing {house_folder}")
        return block_frequency_dict
    # Flatten voxel
    unique_indices, counts = np.unique(voxels, return_counts=True)

    for block_index, count in zip(unique_indices, counts):
        block_name = block_types.get(block_index, "UNKNOWN")
        block_frequency_dict[block_name] = (block_frequency_dict.get(block_name, 0) + int(count))
    del voxels, block_types

    return block_frequency_dict


In [ ]:
from config.paths import HOUSE_3D_CRAFT_DATA_DIR

block_frequency_dict = compute_3DCraft_block_frequency(HOUSE_3D_CRAFT_DATA_DIR, max_size=None)


## Plot block frequency

In [ ]:
import matplotlib.pyplot as plt
import math

block_frequency_dict = dict(sorted(block_frequency_dict.items(), key=lambda item: item[1], reverse=False))
block_frequency_dict_cleaned = {str(key).replace("minecraft:", "").replace("_", "\n"): value for key, value in block_frequency_dict.items()}
_ = block_frequency_dict_cleaned.pop("air")
save_dir_path = PROJECT_ROOT / "notebooks" / "analyse_data_plots"
os.makedirs(save_dir_path, exist_ok=True)

# Plot most frequent block types
plt.figure(figsize=(12, 4))
plt.xlabel("Block Type")
plt.ylabel("Count")
plt.title("15 Most Frequent Block Types in 3D Craft Houses (excluding air)")
plt.bar(list(block_frequency_dict_cleaned.keys())[-15:], list(block_frequency_dict_cleaned.values())[-15:], width=0.5)
plt.grid(True, axis="y")
plt.tight_layout()
plt.savefig(save_dir_path / "least_frequent_block_types.png", dpi=300)
plt.show()

# Plot most frequent block types
plt.figure(figsize=(12, 4))
plt.xlabel("Block Type")
plt.ylabel("Count")
plt.title("15 Least Frequent Block Types in 3D Craft Houses")
plt.bar(list(block_frequency_dict_cleaned.keys())[:15], list(block_frequency_dict_cleaned.values())[:15], width=0.5)
plt.grid(True, axis="y")
plt.tight_layout()
plt.savefig(save_dir_path / "most_frequent_block_types.png", dpi=300)
plt.show()


## Plot Important Block Frequency

In [ ]:
import matplotlib.pyplot as plt
import math

block_frequency_dict_cleaned = {str(key).replace("minecraft:", "").replace("_", "\n"): value for key, value in block_frequency_dict.items()}
_ = block_frequency_dict_cleaned.pop("air")

# Plot block types with important block
important_blocks = [
    # "glass", 
    "stair", 
    "gate", 
    "door"
    ]
num_plots = len(important_blocks)
fig, axes = plt.subplots(num_plots, 1, figsize=(10, 4 * num_plots), squeeze=False)
axes = axes.flatten()

for i, block in enumerate(important_blocks):
    tmp_frequency_dict = {key: value for key, value in block_frequency_dict_cleaned.items() if block in key}
    tmp_chunks = sorted(tmp_frequency_dict.items(), key=lambda x: x[1])
    block_names = [item[0] for item in tmp_chunks]
    counts = [item[1] for item in tmp_chunks]

    ax = axes[i]
    ax.bar(block_names, counts, width=0.4)
    ax.set_title(f"Important blocks in 3D Craft Houses ({block.upper()})")
    ax.set_xlabel("Block Type")
    ax.set_ylabel("Count")
    ax.grid(True, axis="y")

    ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


# Plot Most Used Blocks

In [ ]:
from config.paths import HOUSE_3D_CRAFT_DATA_DIR

# Get the most frequent block type in each house
house_folders = [f for f in os.listdir(HOUSE_3D_CRAFT_DATA_DIR) if f.startswith("workdir")]
house_block_frequencies = {}
for house_folder in house_folders:
    block_freq = compute_3DCraft_block_frequency_individual(HOUSE_3D_CRAFT_DATA_DIR, house_folder)
    block_freq_cleaned = {str(key).replace("minecraft:", "").replace("_", "\n"): value for key, value in block_freq.items()}
    _ = block_freq_cleaned.pop("air", None)
    sorted_blocks = sorted(block_freq_cleaned.items(), key=lambda x: x[1], reverse=True)
    house_block_frequencies[house_folder] = sorted_blocks[:1]



In [ ]:
import matplotlib.pyplot as plt

# Count how many times each block type appears in the top 1 of houses
top_block_counts = {}
for house, blocks in house_block_frequencies.items():
    for block, count in blocks:
        top_block_counts[block] = top_block_counts.get(block, 0) + 1
        
# Plot the most common top blocks across houses
top_block_counts_sorted = dict(sorted(top_block_counts.items(), key=lambda item: item[1], reverse=True))
plt.figure(figsize=(12, 6))
plt.bar(list(top_block_counts_sorted.keys())[:15], list(top_block_counts_sorted.values())[:15], width=0.5)
plt.xlabel("Block Type")
plt.ylabel("Number of Houses with Block in Top 3")
plt.title("Most Common Top 3 Blocks Across 3D Craft Houses")
plt.xticks(rotation=30, ha="left")
plt.grid(True, axis="y")
plt.tight_layout()
# plt.savefig(save_dir_path / "most_common_top_blocks.png", dpi=300)
plt.show()